In [3]:
import pandas as pd
df = pd.read_csv('Subject 1.csv', parse_dates=['EventDateTime'])
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11042 entries, 0 to 11041
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   EventDateTime               11042 non-null  datetime64[ns]
 1   DeviceMode                  0 non-null      float64       
 2   BolusType                   593 non-null    object        
 3   Basal                       7419 non-null   float64       
 4   CorrectionDelivered         593 non-null    float64       
 5   TotalBolusInsulinDelivered  593 non-null    float64       
 6   FoodDelivered               593 non-null    float64       
 7   CarbSize                    593 non-null    float64       
 8   CGM                         11042 non-null  int64         
dtypes: datetime64[ns](1), float64(6), int64(1), object(1)
memory usage: 776.5+ KB


In [4]:
df = df.sort_values('EventDateTime')

In [5]:
df['EventDateTime'] = pd.to_datetime(df['EventDateTime'])
df = df.set_index('EventDateTime')

In [6]:
df['hour'] = df.index.hour
df['day_of_week'] = df.index.dayofweek
df['day_night'] = df['hour'].apply(lambda x: 0 if (x < 6 or x >= 20) else 1)


In [7]:
import numpy as np

df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24)

df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week']/7)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week']/7)



In [11]:
df['FoodDelivered'].fillna(0, inplace=True)
df['TotalBolusInsulinDelivered'].fillna(0,inplace=True)

/var/folders/8r/w2tfcf3d63b2dfp05x4fmyx40000gn/T/ipykernel_5155/2558290490.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['FoodDelivered'].fillna(0, inplace=True)
/var/folders/8r/w2tfcf3d63b2dfp05x4fmyx40000gn/T/ipykernel_5155/2558290490.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beha

In [13]:
df = df.drop(columns=['DeviceMode'])

In [15]:
df['BolusType'] = df['BolusType'].fillna('NoBolus')

In [16]:
df = pd.get_dummies(df, columns=['BolusType'], prefix='Bolus')

In [20]:
df['ActiveBolus'] = df['TotalBolusInsulinDelivered'].rolling(window=24, min_periods=1).sum()

In [21]:
df.head()

,Basal,CorrectionDelivered,TotalBolusInsulinDelivered,FoodDelivered,CarbSize,CGM,hour,day_of_week,day_night,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,Bolus_Automatic Bolus/Correction,Bolus_BLE Standard Bolus/Correction,Bolus_Extended 50.00%/0.00,Bolus_NoBolus,Bolus_Standard,Bolus_Standard/Correction,ActiveBolus
EventDateTime,,,,,,,,,,,,,,,,,,,,
2023-12-08 00:04:00,NaN,NaN,0.0,0.0,NaN,151,0,4,0,0.0,1.0,-0.433884,-0.900969,False,False,False,True,False,False,0.0
2023-12-08 00:09:00,NaN,NaN,0.0,0.0,NaN,152,0,4,0,0.0,1.0,-0.433884,-0.900969,False,False,False,True,False,False,0.0
2023-12-08 00:14:00,NaN,NaN,0.0,0.0,NaN,156,0,4,0,0.0,1.0,-0.433884,-0.900969,False,False,False,True,False,False,0.0
2023-12-08 00:19:00,NaN,NaN,0.0,0.0,NaN,158,0,4,0,0.0,1.0,-0.433884,-0.900969,False,False,False,True,False,False,0.0
2023-12-08 00:24:00,NaN,NaN,0.0,0.0,NaN,160,0,4,0,0.0,1.0,-0.433884,-0.900969,False,False,False,True,False,False,0.0


In [23]:
features = [
    'hour_sin', 'hour_cos',
    'day_of_week_sin', 'day_of_week_cos',
    'day_night',
    'FoodDelivered', 'CarbSize',
    'TotalBolusInsulinDelivered', 'Basal', 'CorrectionDelivered',
    'ActiveBolus', 'Bolus_Automatic Bolus/Correction', 'Bolus_BLE Standard Bolus/Correction', 'Bolus_Extended 50.00%/0.00',
    'Bolus_Standard', 'Bolus_Standard/Correction', 'Bolus_NoBolus'
]

target = 'CGM'

In [27]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df[features] = scaler.fit_transform(df[features])

import numpy as np

X, y = [], []
window_size = 12  # 12 steps back (1 hour)
for i in range(len(df) - window_size):
    X.append(df[features].iloc[i:i+window_size].values)
    y.append(df[target].iloc[i+window_size])  # target after 1 step forward 

X = np.array(X)
y = np.array(y)

print(X.shape)  
print(y.shape)  

(11030, 12, 17)
(11030,)
